In [3]:
# Random Forest is a powerful ensemble learning algorithm 
# used for both classification and regression tasks. 
# It builds multiple decision trees and combines their outputs to improve accuracy 
# and reduce overfitting.
# Many weak learners together become a strong learner
# For classification → majority voting
# For regression → average prediction

# 2. How It Works (Step-by-Step)
# Bootstrap Sampling (Bagging)
# Random subsets of data are created (with replacement)
# Build Decision Trees
# Each subset is used to train a separate tree
# Feature Randomness
# At each split, only a random subset of features is considered
# (this makes trees less correlated)
# Aggregation
# Combine outputs of all trees:
# Voting (classification)
# Averaging (regression)

#Core Libraries
import numpy as np
import pandas as pd

In [6]:
#ML Utilities
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
)

from sklearn.inspection import permutation_importance

In [7]:
#Create Real-World loan Approval Dataset
np.random.seed(42)

n_samples = 1000

df = pd.DataFrame({
    "Age": np.random.randint(21, 65, n_samples),
    "Income": np.random.randint(20000, 150000, n_samples),
    "Credit_Score": np.random.randint(300, 850, n_samples),
    "Loan_Amount": np.random.randint(50000, 800000, n_samples),
    "Employment_Years": np.random.randint(0, 30, n_samples),
    "Loan_Approved": np.random.choice([0, 1], size=n_samples, p=[0.45, 0.55])
})

In [8]:
df.head()

,Age,Income,Credit_Score,Loan_Amount,Employment_Years,Loan_Approved
0,59,24018,505,187888,20,1
1,49,31302,754,127175,21,1
2,35,87506,572,638595,20,1
3,63,61157,535,358877,7,1
4,28,74917,820,145028,16,0


In [9]:
x=df.drop("Loan_Approved", axis=1)  # axis 1 means column 0 means rows
y=df['Loan_Approved']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    x,y,
    test_size=0.25,
    stratify=y,
    random_state=42
) 

In [18]:
x.columns

Index(['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Employment_Years'], dtype='object')

In [15]:
numeric_features= x.columns.tolist()
numeric_features

['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Employment_Years']

In [17]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [19]:
preprocessor = ColumnTransformer(transformers=[('num', numeric_pipeline, numeric_features)])

In [20]:
#Random Forest Model (Advanced Configuration)
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    oob_score=True,
    n_jobs=-1,
    random_state=42   
)  

# n_estimators=300 → Uses 300 decision trees to improve prediction stability
# max_depth=None → Trees grow fully without depth restriction
# min_samples_split=5 → At least 5 samples are needed to split a node
# min_samples_leaf=2 → Each leaf must have at least 2 samples
# max_features="sqrt" → Uses √(features) at each split for randomness
# class_weight="balanced" → Automatically adjusts weights for imbalanced classes
# oob_score=True → Uses out-of-bag samples to estimate model accuracy
# n_jobs=-1 → Uses all CPU cores for faster computation
# random_state=42 → Ensures reproducible results

In [23]:
#Full Pipeline
model_pipeline= Pipeline(steps=[
    ("preprocessing", preprocessor),
    ('model', rf_model)
])

In [24]:
#Model Training
model_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse

In [25]:
#OOB Score
print('OOB Score:', model_pipeline.named_steps['model'].oob_score_)

OOB Score: 0.516


In [26]:
#Predictions
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:,1]  #for poitive labels approved loans

C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\User

In [30]:
# Advanced Evaluation Metrics 
print('\nAdvanced Evaluation Metrics')

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1-Score:', f1_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))

print('\nClassification Report')
print(classification_report(y_test, y_pred))

print('\nConfusion Matrix')
print(confusion_matrix(y_test, y_pred))


Advanced Evaluation Metrics
Accuracy: 0.508
Precision: 0.5289855072463768
Recall: 0.5572519083969466
F1-Score: 0.5427509293680297
ROC-AUC: 0.5183783437038938

Classification Report
              precision    recall  f1-score   support

           0       0.48      0.45      0.47       119
           1       0.53      0.56      0.54       131

    accuracy                           0.51       250
   macro avg       0.51      0.51      0.51       250
weighted avg       0.51      0.51      0.51       250


Confusion Matrix
[[54 65]
 [58 73]]


In [31]:
#Feature Importance
feature_names= numeric_features
importances = model_pipeline.named_steps['model'].feature_importances_

feature_importance_df=pd.DataFrame({
    "Feature": feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)
print('\nFeature Importance (Gini)')
print(feature_importance_df)
      

C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\User


Feature Importance (Gini)
            Feature  Importance
1            Income    0.222618
3       Loan_Amount    0.222400
2      Credit_Score    0.214975
0               Age    0.182423
4  Employment_Years    0.157584


C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\User

In [ ]:
#Permutation Importance (Model-Agnostic)
perm_importance=permutation_importance(
    model_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)
perm_df=pd.DataFrame({
    "Feature": feature_names,
    

In [44]:
# Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []

for train_idx, val_idx in skf.split(x, y):
    X_tr, X_val = x.iloc[train_idx], x.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_pipeline.fit(X_tr, y_tr)
    preds = model_pipeline.predict(X_val)

    cv_scores.append(f1_score(y_val, preds))

print('\nStratified CV F1 Scores:', cv_scores)
print('Mean CV F1 Score:', np.mean(cv_scores))


Stratified CV F1 Scores: [0.5688888888888889, 0.5701754385964912, 0.5192307692307693, 0.6060606060606061, 0.5495495495495496]
Mean CV F1 Score: 0.562781050465261


In [47]:
#Advanced Hyperparamter Tuning (Randomized Search)
param_dist = {
    "model__n_estimators": [200,300,500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2,5,10],
    "model__min_samples_leaf": [1,2,4],
    "model__max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    model_pipeline,
    param_distributions= param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strat

In [49]:
#Best Model After Tuning
best_model = random_search.best_estimator_
print('\nBest Hyperparameters')
print(random_search.best_params_)


Best Hyperparameters
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 30}


In [53]:
#Final Tuned Model Evaluation
final_pred = best_model.predict(X_test)
final_prob= best_model.predict_proba(X_test)[:,1]
print('\nFinal Tuned Model Performance')
print('Accuracy:', accuracy_score(y_test, final_pred))
print('Precision:', precision_score(y_test, final_pred))
print('Recall:', recall_score(y_test, final_pred))
print('F1-Score:', f1_score(y_test, final_pred))
print('ROC-AUC:',roc_auc_score(y_test, final_prob))




Final Tuned Model Performance
Accuracy: 0.524
Precision: 0.5461538461538461
Recall: 0.5419847328244275
F1-Score: 0.5440613026819924
ROC-AUC: 0.52152158573353
